# Notebook 01 — How VLMs See Images

Before we build anything multimodal, we need to know what an image **looks like to a VLM**, because that determines two things every senior engineer is asked about:

1. **Quality** — what details survive the encoding?
2. **Cost** — how many tokens does an image consume?

## The mental model

From the LLM's perspective, an image is just **more tokens**. The work is in producing those tokens. A modern VLM does it in three steps (S3 §1.2):

```
image  →  Vision Transformer (patches)  →  Projector  →  visual tokens  →  LLM
```

The Vision Transformer (ViT) chops an image into a grid of square patches (typically 14×14 or 16×16 pixels), turns each patch into one vector, and adds 2D positional embeddings. A 336×336 image with 14×14 patches becomes a **24×24 = 576 patch grid** — i.e., 576 visual tokens that get prepended to the text tokens.

**Key insight:** image token count is a deterministic function of image dimensions. You can compute the bill before you make the call.

Let's see this concretely.

## 1. Visualize ViT patching

We'll grab any image and overlay a 14×14 patch grid so you can *see* what the encoder sees.

In [ ]:
import io
import urllib.request
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# A simple test image — swap this for any local file you like.
URL = "https://images.unsplash.com/photo-1574158622682-e40e69881006?w=640"
img_bytes = urllib.request.urlopen(URL).read()
img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

# CLIP-ViT-L/14 expects 336×336 input (S3 §1.3)
img_336 = img.resize((336, 336))

PATCH = 14
n_patches_per_side = 336 // PATCH
total_patches = n_patches_per_side ** 2
print(f"At 336×336 with {PATCH}×{PATCH} patches → {n_patches_per_side}×{n_patches_per_side} grid → {total_patches} visual tokens")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img_336)
for i in range(n_patches_per_side + 1):
    ax.axhline(i * PATCH, color="white", lw=0.3, alpha=0.6)
    ax.axvline(i * PATCH, color="white", lw=0.3, alpha=0.6)
ax.set_title(f"ViT-L/14 sees this as {total_patches} patch tokens")
ax.set_axis_off()
plt.show()

Each white-bordered square becomes one ~1024-dim vector after the ViT's linear projection + transformer layers. The projector then maps each one to the LLM's embedding dimension (e.g., 4096 for Llama-3-8B). For an MLP projector (LLaVA-style), **patch count = visual token count**.

**Why this is important:** double the resolution → 4× the tokens. Quadratic scaling is why image cost spirals fast on dense documents.

## 2. The GPT-4o image-token formula

OpenAI bills by image tokens, not bytes. The formula (S3 §2.1, [official docs](https://platform.openai.com/docs/guides/images-vision)) as of 2026:

```
detail = "low":   cost = 85 tokens, regardless of image size (downsampled to 512×512)

detail = "high":  1. Scale image to fit in 2048×2048 box (preserve aspect)
                  2. Scale so the shortest side is 768 px
                  3. Count 512×512 tiles needed to cover the image
                  4. cost = 85 + 170 × num_tiles
```

Let's implement it.

In [ ]:
import math

def gpt4o_image_tokens(width: int, height: int, detail: str = "high") -> int:
    """Returns the number of input tokens GPT-4o will charge for an image."""
    if detail == "low":
        return 85

    # Step 1: fit in 2048×2048 box
    if max(width, height) > 2048:
        scale = 2048 / max(width, height)
        width, height = int(width * scale), int(height * scale)

    # Step 2: shortest side → 768
    short = min(width, height)
    if short > 768:
        scale = 768 / short
        width, height = int(width * scale), int(height * scale)

    # Step 3: count 512×512 tiles
    tiles_w = math.ceil(width / 512)
    tiles_h = math.ceil(height / 512)
    return 85 + 170 * tiles_w * tiles_h


# Sanity check against the worked examples in the OpenAI docs
assert gpt4o_image_tokens(1024, 1024, "high") == 765    # 4 tiles
assert gpt4o_image_tokens(2048, 4096, "high") == 1105   # 6 tiles
assert gpt4o_image_tokens(8000, 8000, "low") == 85
print("All assertions passed.")

## 3. Plot tokens vs resolution

Build the intuition by plotting the curve.

In [ ]:
import numpy as np

sizes = [256, 512, 768, 1024, 1280, 1536, 2048, 3000, 4096]
high = [gpt4o_image_tokens(s, s, "high") for s in sizes]
low = [gpt4o_image_tokens(s, s, "low") for s in sizes]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, high, "o-", label="detail=high")
ax.plot(sizes, low, "s--", label="detail=low")
ax.set_xlabel("Image side length (px, square)")
ax.set_ylabel("Input tokens charged")
ax.set_title("GPT-4o image tokens vs resolution")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

for s, h, l in zip(sizes, high, low):
    print(f"{s:>4}×{s:<4}  high={h:>5}  low={l:>3}")

Notice that beyond ~768 px, **the curve plateaus** — that's the shortest-side-→-768 rescale kicking in. Sending a 4096×4096 phone photo charges the same as a 1024×1024 image. **Resizing before send is free money.**

## 4. Translate to dollars

GPT-4o-mini input pricing as of early 2026: **$0.15 / 1M input tokens**. GPT-4o: **$2.50 / 1M**.

In [ ]:
PRICING = {
    "gpt-4o-mini": 0.15 / 1_000_000,    # $/token
    "gpt-4o":      2.50 / 1_000_000,
}

scenarios = [
    ("Phone snap (4032×3024)",       4032, 3024, "high"),
    ("Resized phone snap (1024×768)", 1024,  768, "high"),
    ("Same, detail=low",              4032, 3024, "low"),
    ("Document scan (2480×3508)",    2480, 3508, "high"),
    ("Profile thumbnail",              256,  256, "low"),
]

print(f"{'Scenario':<35} {'tokens':>8}  {'mini':>10}  {'4o':>10}")
print("-" * 70)
for name, w, h, d in scenarios:
    tok = gpt4o_image_tokens(w, h, d)
    mini_cost = tok * PRICING["gpt-4o-mini"]
    full_cost = tok * PRICING["gpt-4o"]
    print(f"{name:<35} {tok:>8}  ${mini_cost:>8.6f}  ${full_cost:>8.6f}")

Per-image cost looks tiny — but multiply by 1M images/day and the order of magnitude matters. **The two highest-leverage cost wins are (1) resize before send, and (2) `detail="low"` for non-OCR tasks.**

## 5. Exercise

Find an image on your laptop. Open it, read its dimensions, and predict the token cost at `detail="high"` *before* you send it. Then send it via the OpenAI API and compare against `response.usage.prompt_tokens`. They should match within ~50 tokens (the system prompt accounts for the rest).

If they don't match, the most likely culprits are:
- An old API version with different patch geometry
- The image was auto-rotated by EXIF before you measured it
- You're using GPT-4.1 / GPT-5, which use 32×32 patches with a 1536-token cap (S3 §2.1)

## What we learned

- A VLM "sees" an image as a grid of patch tokens — typically a few hundred to a few thousand per image.
- GPT-4o's token count is a deterministic function of dimensions + `detail`. You can predict cost before sending.
- Resolution scales tokens **quadratically**. Resize aggressively.
- `detail="low"` flat-rate is your friend for coarse tasks ("is there a person here"); reserve `detail="high"` for OCR / dense charts.

**Next:** [Notebook 02 — CLIP text→image search](02_clip_text_to_image_search.ipynb). Once we know how images become tokens, we can ask: *can we put image tokens and text tokens in the same vector space and search across them?*